In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import to_date, date_format


def create_gold_dim_salesperson(employee_tbl_df, salesperson_tbl_df, person_tbl_df):

    salesperson_tbl_df = salesperson_tbl_df.select("BusinessEntityID","TerritoryID","SalesQuota","Bonus","CommissionPct")
    employee_tbl_df = employee_tbl_df.select("BusinessEntityID","HireDate","JobTitle")
    person_tbl_df = person_tbl_df.select("BusinessEntityID", "PersonID", "FirstName", "LastName")
    dim_salesperson_tgt_df = salesperson_tbl_df.join(employee_tbl_df, salesperson_tbl_df.BusinessEntityID == employee_tbl_df.BusinessEntityID, "inner").drop(employee_tbl_df.BusinessEntityID)
    
    dim_salesperson_tgt_df = dim_salesperson_tgt_df.join(person_tbl_df,  dim_salesperson_tgt_df.BusinessEntityID == person_tbl_df.BusinessEntityID, "inner").drop(person_tbl_df.BusinessEntityID).dropDuplicates(["BusinessEntityID"]).withColumn("processed_timestamp", F.current_timestamp())
                                 
    return dim_salesperson_tgt_df



if __name__ == "__main__":

    employee_tbl = dbutils.widgets.get("employee")
    salesperson_tbl = dbutils.widgets.get("salesperson")
    person_tbl = dbutils.widgets.get("person")
    employee_tbl_df = df = spark.read.table(employee_tbl)
    salesperson_tbl_df = df = spark.read.table(salesperson_tbl)
    person_tbl_df = df = spark.read.table(person_tbl)

    dim_salesperson_tgt_df = create_gold_dim_salesperson(employee_tbl_df, salesperson_tbl_df, person_tbl_df)
    dim_salesperson_tbl = dbutils.widgets.get("dim_salesperson")
    dim_salesperson_tgt_df.write.mode("overwrite").format("delta").partitionBy("BusinessEntityID").saveAsTable(dim_salesperson_tbl)